In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2
%aimport -numpy
%aimport -pandas
%aimport -matplotlib
%aimport -seaborn
%aimport -scipy
%aimport -sklearn

print(f"Raiz do projeto: {project_root}")

Raiz do projeto: /data_lids/home/gabrielseabra/credit-score-analysis


## 1. Carregamento dos Dados

In [4]:
import pandas as pd
import numpy as np

from src.data.loader import HomeCreditDataLoader

loader = HomeCreditDataLoader(data_dir="../data")
df_train = loader.load_application_train()

print(f"Shape: {df_train.shape}")
df_train.head(3)

2026-03-10 14:41:32,593 - HomeCreditDataLoader - INFO - Carregando dados de: application_train.csv...
2026-03-10 14:41:32,724 - HomeCreditDataLoader - INFO - Sucesso! application_train.csv carregado com formato (307511, 122).


Shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Prototipagem — AnomalyHandler

Antes de usar a classe, validamos a lógica manualmente.

In [5]:
# Verificação antes
print("DAYS_EMPLOYED — anomalias antes do tratamento:")
print(f"  Valor 365243 presente: {(df_train['DAYS_EMPLOYED'] == 365243).sum():,} linhas")
print(f"  NaN presentes        : {df_train['DAYS_EMPLOYED'].isna().sum():,} linhas")

DAYS_EMPLOYED — anomalias antes do tratamento:
  Valor 365243 presente: 55,374 linhas
  NaN presentes        : 0 linhas


In [6]:
from src.features.build_features import AnomalyHandler

handler = AnomalyHandler()
df_clean = handler.fit_transform(df_train)

# Verificação depois
print("DAYS_EMPLOYED — após AnomalyHandler:")
print(f"  Valor 365243 presente    : {(df_clean['DAYS_EMPLOYED'] == 365243).sum():,} linhas")
print(f"  NaN presentes            : {df_clean['DAYS_EMPLOYED'].isna().sum():,} linhas")
print(f"  DAYS_EMPLOYED_ANOMALY=1  : {df_clean['DAYS_EMPLOYED_ANOMALY'].sum():,} linhas")
print()
print("Flag criada corretamente:", df_clean['DAYS_EMPLOYED_ANOMALY'].value_counts().to_dict())
print()
# Garante que o original não foi modificado (sem side-effects)
assert (df_train['DAYS_EMPLOYED'] == 365243).sum() > 0, "Original foi modificado — side-effect indesejado!"
print("OK — DataFrame original não foi modificado.")

DAYS_EMPLOYED — após AnomalyHandler:
  Valor 365243 presente    : 0 linhas
  NaN presentes            : 55,374 linhas
  DAYS_EMPLOYED_ANOMALY=1  : 55,374 linhas

Flag criada corretamente: {0: 252137, 1: 55374}

OK — DataFrame original não foi modificado.


## 3. Prototipagem — DomainFeatureBuilder

In [7]:
from src.features.build_features import DomainFeatureBuilder

builder = DomainFeatureBuilder()
df_enriched = builder.fit_transform(df_clean)

# Features criadas
new_features = [
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM_MONTHS',
    'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_TO_AGE_RATIO', 'INCOME_PER_FAMILY_MEMBER'
]

print("Documentação das features criadas:")
print(DomainFeatureBuilder.feature_descriptions().to_string(index=False))
print()
print("Primeiras linhas:")
df_enriched[new_features].head()

Documentação das features criadas:
                 feature                                                                 descricao
     CREDIT_INCOME_RATIO        Crédito total / Renda anual. Razão alta indica superendividamento.
    ANNUITY_INCOME_RATIO            Parcela mensal / Renda anual. Mede o comprometimento de renda.
      CREDIT_TERM_MONTHS                Crédito / Parcela. Prazo implícito do empréstimo em meses.
INCOME_PER_FAMILY_MEMBER        Renda / Membros da família. Proxy de capacidade de pagamento real.
               AGE_YEARS                            Idade do cliente convertida de dias para anos.
          EMPLOYED_YEARS Tempo de emprego convertido de dias para anos (após limpeza de anomalia).
   EMPLOYED_TO_AGE_RATIO                Tempo empregado / Idade. Estabilidade relativa no emprego.

Primeiras linhas:


,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_TERM_MONTHS,AGE_YEARS,EMPLOYED_YEARS,EMPLOYED_TO_AGE_RATIO,INCOME_PER_FAMILY_MEMBER
0,2.007889,0.121978,16.461104,25.902806,1.744011,0.067329,202500.0
1,4.790750,0.132217,36.234085,45.900068,3.252567,0.070862,135000.0
2,2.000000,0.100000,20.000000,52.145106,0.616016,0.011814,67500.0
3,2.316167,0.219900,10.532818,52.032854,8.320329,0.159905,67500.0
4,4.222222,0.179963,23.461618,54.570842,8.317591,0.152418,121500.0


In [8]:
# Validação: sem NaN inesperados nas novas features (exceto onde esperado)
print("Nulos nas novas features:")
print(df_enriched[new_features].isnull().sum().to_string())
print()
print("Estatísticas descritivas:")
df_enriched[new_features].describe().round(2)

Nulos nas novas features:
CREDIT_INCOME_RATIO             0
ANNUITY_INCOME_RATIO           12
CREDIT_TERM_MONTHS             12
AGE_YEARS                       0
EMPLOYED_YEARS              55374
EMPLOYED_TO_AGE_RATIO       55374
INCOME_PER_FAMILY_MEMBER        2

Estatísticas descritivas:


,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_TERM_MONTHS,AGE_YEARS,EMPLOYED_YEARS,EMPLOYED_TO_AGE_RATIO,INCOME_PER_FAMILY_MEMBER
count,307511.00,307499.00,307499.00,307511.00,252137.00,252137.00,307509.00
mean,3.96,0.18,21.61,43.91,6.53,0.16,93105.88
std,2.69,0.09,7.82,11.95,6.40,0.13,101373.36
min,0.00,0.00,8.04,20.50,-0.00,-0.00,2812.50
25%,2.02,0.11,15.61,33.98,2.10,0.06,47250.00
50%,3.27,0.16,20.00,43.12,4.51,0.12,75000.00
75%,5.16,0.23,27.10,53.89,8.69,0.22,112500.00
max,84.74,1.88,45.31,69.07,49.04,0.73,39000000.00


## 4. Pipeline Completo — Teste End-to-End

Monta o pipeline com `build_preprocessor_pipeline()` e valida que o output
é um array numpy pronto para um modelo de ML.

In [9]:
from src.features.build_features import build_preprocessor_pipeline

# Define as colunas que entrarão no modelo (após o DomainFeatureBuilder criar as novas)
NUMERIC_COLS = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM_MONTHS',
    'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_TO_AGE_RATIO',
    'INCOME_PER_FAMILY_MEMBER', 'DAYS_EMPLOYED_ANOMALY',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
]

CATEGORICAL_COLS = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
]

pipeline = build_preprocessor_pipeline(
    numeric_cols=NUMERIC_COLS,
    categorical_cols=CATEGORICAL_COLS,
)

print(pipeline)

Pipeline(steps=[('anomaly_handler', AnomalyHandler()),
                ('domain_features', DomainFeatureBuilder()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['AMT_INCOME_TOTAL',
                                                   'AMT_CREDIT', 'AMT_ANNUITY',
                                                   'AMT_GOODS_PRICE',
                                                   'CREDIT_INCOME_RATIO',
                                                   'ANNUITY_INCOME_RATIO...
                                                   'CNT_CHILDREN',
           

In [10]:
X = df_train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_train['TARGET']

X_transformed = pipeline.fit_transform(X, y)

print(f"Shape de entrada  : {X.shape}")
print(f"Shape de saída    : {X_transformed.shape}")
print(f"Tipo              : {type(X_transformed)}")
print(f"NaN no output     : {np.isnan(X_transformed).sum()}")
print()
print("Pipeline pronto para ser passado a um modelo de ML.")

Shape de entrada  : (307511, 120)
Shape de saída    : (307511, 69)
Tipo              : <class 'numpy.ndarray'>
NaN no output     : 0

Pipeline pronto para ser passado a um modelo de ML.


## 5. Resumo das Transformações: Colunas de Entrada → Colunas de Saída

O pipeline reduz o DataFrame de **120 colunas de entrada** para **69 colunas de saída** em três etapas.

---

### Etapa 1 — `AnomalyHandler`

**Coluna afetada:** `DAYS_EMPLOYED`

O valor `365243` é um marcador de anomalia presente em 55.374 linhas (clientes sem vínculo empregatício). A transformação:
- Substitui `365243` por `NaN` em `DAYS_EMPLOYED`
- Cria a flag binária `DAYS_EMPLOYED_ANOMALY` (`1` = era anomalia, `0` = valor válido)

| Coluna original | Coluna de saída | Mudança |
|---|---|---|
| `DAYS_EMPLOYED` | `DAYS_EMPLOYED` | 365243 → NaN |
| *(nova)* | `DAYS_EMPLOYED_ANOMALY` | Flag binária indicando a anomalia |

---

### Etapa 2 — `DomainFeatureBuilder`

Cria **7 novas features** a partir de colunas existentes:

| Feature criada | Fórmula | Interpretação |
|---|---|---|
| `CREDIT_INCOME_RATIO` | `AMT_CREDIT / AMT_INCOME_TOTAL` | Nível de endividamento relativo à renda |
| `ANNUITY_INCOME_RATIO` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | Comprometimento mensal da renda |
| `CREDIT_TERM_MONTHS` | `AMT_CREDIT / AMT_ANNUITY` | Prazo implícito do empréstimo em meses |
| `INCOME_PER_FAMILY_MEMBER` | `AMT_INCOME_TOTAL / CNT_FAM_MEMBERS` | Capacidade de pagamento real por pessoa |
| `AGE_YEARS` | `DAYS_BIRTH / -365` | Idade do cliente em anos |
| `EMPLOYED_YEARS` | `DAYS_EMPLOYED / -365` | Tempo de emprego em anos (após limpeza da anomalia) |
| `EMPLOYED_TO_AGE_RATIO` | `EMPLOYED_YEARS / AGE_YEARS` | Estabilidade de emprego relativa à idade |

---

### Etapa 3 — `ColumnTransformer`

Das 128 colunas disponíveis após as etapas anteriores, **somente 26 são selecionadas** para o modelo (as demais — flags de documentos, variáveis de bureau de crédito, etc. — são descartadas).

**Numéricas (17 colunas):** imputação por mediana + `StandardScaler` → saem normalizadas (média 0, desvio padrão 1), sem NaN.

**Categóricas (9 colunas):** imputação por moda + `OneHotEncoder` → expandidas em colunas binárias, totalizando **52 colunas** no output.

---

### Visão geral do pipeline

| Etapa | Colunas de entrada | Colunas de saída | O que muda |
|---|---|---|---|
| `AnomalyHandler` | 120 | 121 | +1 flag `DAYS_EMPLOYED_ANOMALY`; anomalias viram NaN |
| `DomainFeatureBuilder` | 121 | 128 | +7 features de domínio derivadas |
| `ColumnTransformer` | 128 | **69** | Seleciona 26 colunas; normaliza numéricas; expande categóricas em one-hot |